Results in this notebook are incorrect since I use $H \otimes x$ instead of $H \otimes (x + iy)$ for the extended swap test.

In [1]:
import numpy as np
import pickle
from utils_ferm import (
    op_action_tz
)
from utils_CSF_and_UCSF import (
    orthogonal_transform_obt_tbt,
    obt_phys_spatial_to_spin,
    tbt_phys_spatial_to_spin,
    make_short_H_ferm_op
)

from openfermion import (
    FermionOperator,
    jordan_wigner,
    QubitOperator,
    get_sparse_operator,
    normal_ordered
)

from utils_states import (
    somos_to_seniority_config,
    convert_TZ_format_to_sparse_format,
    variance_of_decomp,
    create_composite_state
)

from utils_partitioning import (
    sorted_insertion_decomposition,
    augment_decomp_with_pauli_x
)

filename = 'data/h2o_data.dump'

with open(filename,'rb') as f:
    (
    list_CSF,
    list_list_ia_CSF,
    list_list_theta_CSF,
    list_sym_CSF_vec,
    list_UCSF_tz,
    tz_states,
    somos_list,
    psi_GS_UCSF_smik,
    list_orb_rot,
    x_orbrot,
    Enuc,
    obt_spatial,
    tbt_spatial
    ) = pickle.load(f)

#Rotate orbitals 
if len(list_orb_rot) != 0:
    obt, tbt = orthogonal_transform_obt_tbt(x_orbrot,list_orb_rot,obt_spatial,tbt_spatial)
else:
    obt = obt_phys_spatial_to_spin(obt_spatial)
    tbt = tbt_phys_spatial_to_spin(tbt_spatial)

Nqubits = obt.shape[0]
Norb    = Nqubits // 2
dim     = 2 ** Nqubits
Nstates = len(tz_states)

print(f"Number of qubits : {Nqubits}")

# generate Hamiltonian

Hferm           = make_short_H_ferm_op(0, obt, tbt)
Hqub            = jordan_wigner(Hferm)
qubit_constant  = Hqub.constant
Hqub           -= qubit_constant
Hqub.compress()
qwc_decomp      = sorted_insertion_decomposition(Hqub, 'qwc')
qwc_decomp_swap = augment_decomp_with_pauli_x(qwc_decomp, Nqubits)

# generate state information

configs      = [somos_to_seniority_config(somos, Norb) for somos in somos_list]
states_full  = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in tz_states]

results = {}

for i in range(Nstates):
    for j in range(Nstates):
        if i >= j:

            bra_tz     = tz_states[i]
            bra        = states_full[i]
            bra_somos  = somos_list[i]
            bra_config = configs[i]

            ket_tz     = tz_states[j]
            ket        = states_full[j]
            ket_somos  = somos_list[j]
            ket_config = configs[j]
            
            Heff_ferm = FermionOperator()
            for term, coef in Hferm.terms.items():
                term_on_ket    = op_action_tz(FermionOperator(term), ket_tz[0], ket_tz[1], ket_tz[2])
                matrix_element = bra.dot(convert_TZ_format_to_sparse_format(dim, term_on_ket).T)
                if np.abs(matrix_element) > 1e-10:
                    Heff_ferm += coef * FermionOperator(term)

            Heff_qub       = jordan_wigner(Heff_ferm)
            Heff_qub      -= Heff_qub.constant
            Heff_qub.compress()
            qwc_decomp_eff = sorted_insertion_decomposition(Heff_qub, 'qwc')

            if i == j:
                var_decomp_eff = variance_of_decomp(qwc_decomp_eff, ket, Nqubits, general=True)
                var_decomp     = variance_of_decomp(qwc_decomp    , ket, Nqubits, general=True)

            else:
                composite_state     = create_composite_state(bra, ket, Nqubits)
                qwc_decomp_eff_swap = augment_decomp_with_pauli_x(qwc_decomp_eff, Nqubits)
                var_decomp_eff      = variance_of_decomp(qwc_decomp_eff_swap, composite_state, Nqubits + 1, general=True)
                var_decomp          = variance_of_decomp(qwc_decomp_swap    , composite_state, Nqubits + 1, general=True)

            results[(i,j)] = (bra_config, ket_config, var_decomp_eff, var_decomp)
            print("{:<2} {:<4} {:<10} {:<15}".format(i, j, np.real(np.round(var_decomp_eff, 4)), np.real(np.round(var_decomp, 4))))


Number of qubits : 14
0  0    0.415      36.6772        
1  0    0.2345     1103.1489      
1  1    0.2163     35.3914        
2  0    0.3788     1092.2494      
2  1    0.3756     1021.4504      
2  2    0.2893     40.3724        
3  0    0.3992     1133.1924      
3  1    0.2027     1067.7411      
3  2    0.2418     1056.5622      
3  3    0.8394     44.3712        
4  0    0.1318     1151.9035      
4  1    0.7991     1087.0993      
4  2    0.2449     1076.0682      
4  3    0.1772     1121.69        
4  4    0.4071     39.2445        
5  0    0.2044     1084.1906      
5  1    0.4471     1002.5489      
5  2    0.1849     999.7222       
5  3    0.3032     1046.3838      
5  4    0.2813     1066.8098      
5  5    0.0514     33.1888        
6  0    0.2044     1086.4395      
6  1    1.3567     1011.4192      
6  2    0.1875     1002.1913      
6  3    0.2992     1048.5355      
6  4    0.2818     1068.8642      
6  5    0.1378     990.1376       
6  6    0.007      34.5521       

In [ ]:
variance_list_og          = np.array([list(results.values())[i][3] for i in range(len(results.values()))])
variance_og_neyman        = np.sum(variance_list_og ** (1/2)) ** 2

variance_list_processed   = np.array([list(results.values())[i][2] for i in range(len(results.values()))])
variance_processed_neyman = np.sum(variance_list_processed ** (1/2)) ** 2

print(f'''
    Estimator variance using original Hamiltonian                     : {variance_og_neyman}
    Estimator variance using processed Hamiltonians                   : {variance_processed_neyman}

    Number of shots to chemical accuracy using original Hamiltonian   : {variance_og_neyman / 1e-6}
    Number of shots to chemical accuracy using processed Hamiltonians : {variance_processed_neyman / 1e-6}
''')


    Estimator variance using original Hamiltonian                     : (3436005.720986499+0j)
    Estimator variance using processed Hamiltonians                   : (1075.5688096319197+9.027329765875508e-07j)

    Number of shots to chemical accuracy using original Hamiltonian   : (3436005720986.499+0j)
    Number of shots to chemical accuracy using processed Hamiltonians : (1075568809.6319199+0.9027329765875508j)




In [19]:
3436005720986.499 / 1075568809.6319199

3194.5940512744746